<a href="https://colab.research.google.com/github/kimkevy/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np, pandas as pd

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({

"agent_id": [f"AG{i:03d}" for i in range(1, 21)],

"county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),

"onboarded": pd.date_range("2024-01-01", periods=20, freq="MS")

    .strftime("%Y-%m-%d"),

})

txns = pd.DataFrame({

"txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],

"agent_id": rng.choice(agents["agent_id"], n),

"txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ",

        "Withdrawal", "deposit"], n),

"amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),

"txn_date": (pd.to_datetime("2026-03-01")

          + pd.to_timedelta(rng.integers(0, 60, n), unit="D"))

          .strftime("%Y-%m-%d"),

})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan

txns.loc[idx[15:20], "amount"] = "N/A"

txns.loc[idx[20:25], "amount"] = "250000"

txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)

txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)

txns.to_csv("agent_transactions.csv", index=False)

In [12]:
import os

# Create the outputs directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)


In [13]:
%%writefile cleaning_pipeline.py
import pandas as pd
import numpy as np

def clean_agent_data(txns_df, agents_df):
    """
    Cleans the mobile money agent network transaction data and merges it with the agent register.

    Parameters:
        txns_df (pd.DataFrame): The raw transaction DataFrame.
        agents_df (pd.DataFrame): The agent register DataFrame.

    Returns:
        clean_df (pd.DataFrame): The cleaned and merged DataFrame.
        log (dict): A summary log of the cleaning steps and metrics.
    """
    log = {}
    df_tx = txns_df.copy()
    df_ag = agents_df.copy()

    # 1. Uniqueness: Identify and remove duplicate transactions
    initial_rows = len(df_tx)
    df_tx = df_tx.drop_duplicates(subset=['txn_id'])
    duplicates_removed = initial_rows - len(df_tx)
    log['duplicates_removed'] = duplicates_removed

    # 2. Consistency: Standardise txn_type into exactly "Deposit" and "Withdrawal"
    df_tx['txn_type'] = df_tx['txn_type'].astype(str).str.strip().str.capitalize()
    # Standardize names to 'Deposit' and 'Withdrawal'
    # Any capitalization mismatch is already standard now. Let's make sure they map correctly
    df_tx['txn_type'] = df_tx['txn_type'].replace({'Deposit ': 'Deposit', 'Deposit': 'Deposit', 'Withdrawal': 'Withdrawal'})

    # 3. Validity and types: Convert amount to numeric and txn_date to datetime
    # First handle amount format problems (N/A, NaNs, etc.)
    df_tx['amount'] = df_tx['amount'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
    df_tx['amount'] = pd.to_numeric(df_tx['amount'], errors='coerce')

    # 4. Completeness: Handle missing and invalid amounts
    # Flag imputed values
    df_tx['amount_imputed'] = df_tx['amount'].isna()
    # We will impute missing amounts with the median amount of that transaction type
    deposit_median = df_tx[df_tx['txn_type'] == 'Deposit']['amount'].median()
    withdrawal_median = df_tx[df_tx['txn_type'] == 'Withdrawal']['amount'].median()

    df_tx.loc[df_tx['amount'].isna() & (df_tx['txn_type'] == 'Deposit'), 'amount'] = deposit_median
    df_tx.loc[df_tx['amount'].isna() & (df_tx['txn_type'] == 'Withdrawal'), 'amount'] = withdrawal_median
    # Default fallback
    df_tx['amount'] = df_tx['amount'].fillna(df_tx['amount'].median())

    # Convert txn_date to datetime, handling format problems (like YYYY/MM/DD and YYYY-MM-DD)
    df_tx['txn_date'] = df_tx['txn_date'].astype(str).str.replace('/', '-')
    df_tx['txn_date'] = pd.to_datetime(df_tx['txn_date'], errors='coerce')

    # 5. Outliers: Use IQR method to flag suspicious amounts
    q1 = df_tx['amount'].quantile(0.25)
    q3 = df_tx['amount'].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df_tx['is_outlier'] = (df_tx['amount'] < lower_bound) | (df_tx['amount'] > upper_bound)

    # 6. Merge: Join transactions to the agent register
    # Report how many transactions reference unknown agents
    unknown_agents_mask = ~df_tx['agent_id'].isin(df_ag['agent_id'])
    unknown_agents_count = unknown_agents_mask.sum()
    log['unknown_agents_count'] = unknown_agents_count

    # Perform left merge
    merged_df = pd.merge(df_tx, df_ag, on='agent_id', how='left')

    # Fill missing county details for unknown agents
    merged_df['county'] = merged_df['county'].fillna('Unknown')

    log['final_row_count'] = len(merged_df)
    return merged_df, log


Overwriting cleaning_pipeline.py


In [14]:
import cleaning_pipeline

# Perform a quick run-through and validation of the cleaning pipeline
clean_df, log = cleaning_pipeline.clean_agent_data(txns, agents)
print("Cleaning log summary:", log)
display(clean_df.head())

# Save clean data
clean_df.to_csv('outputs/agent_transactions_clean.csv', index=False)


Cleaning log summary: {'duplicates_removed': 12, 'unknown_agents_count': np.int64(7), 'final_row_count': 600}


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,onboarded
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,2024-09-01
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,2025-08-01
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,2024-08-01
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,2024-09-01
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,2024-09-01


In [15]:
import pandas as pd
import numpy as np

audit_records = []

duplicate_rows = txns.duplicated(subset=['txn_id']).sum()
audit_records.append({
    'Issue': 'Duplicate transaction IDs found',
    'Dimension': 'Uniqueness',
    'Column': 'txn_id',
    'Rows Affected': duplicate_rows
})

inconsistent_types_count = txns[~txns['txn_type'].isin(['Deposit', 'Withdrawal'])].shape[0]
audit_records.append({
    'Issue': 'Inconsistent transaction type formats (whitespace or mixed-case)',
    'Dimension': 'Consistency',
    'Column': 'txn_type',
    'Rows Affected': inconsistent_types_count
})

missing_or_na_amounts = txns['amount'].isna().sum() + (txns['amount'] == 'N/A').sum()
audit_records.append({
    'Issue': 'Missing or "N/A" values in transaction amounts',
    'Dimension': 'Completeness',
    'Column': 'amount',
    'Rows Affected': missing_or_na_amounts
})

date_format_issues = txns['txn_date'].str.contains('/').sum()
audit_records.append({
    'Issue': 'Dates formatted with "/" instead of "-"',
    'Dimension': 'Validity',
    'Column': 'txn_date',
    'Rows Affected': date_format_issues
})

unknown_agents_total = (~txns['agent_id'].isin(agents['agent_id'])).sum()
audit_records.append({
    'Issue': 'Transactions referencing agent IDs not in registration list',
    'Dimension': 'Validity',
    'Column': 'agent_id',
    'Rows Affected': unknown_agents_total
})

audit_df = pd.DataFrame(audit_records)
display(audit_df)

,Issue,Dimension,Column,Rows Affected
0,Duplicate transaction IDs found,Uniqueness,txn_id,12
1,Inconsistent transaction type formats (whitesp...,Consistency,txn_type,384
2,"Missing or ""N/A"" values in transaction amounts",Completeness,amount,22
3,"Dates formatted with ""/"" instead of ""-""",Validity,txn_date,5
4,Transactions referencing agent IDs not in regi...,Validity,agent_id,8


In [16]:
outliers_only = clean_df[clean_df['is_outlier']]
print(f"Number of outliers flagged (IQR method): {len(outliers_only)}")
print("Investigation of KES 250,000 values:")
print(clean_df[clean_df['amount'] == 250000])

Number of outliers flagged (IQR method): 53
Investigation of KES 250,000 values:
      txn_id agent_id    txn_type    amount   txn_date  amount_imputed  \
306  TX00307    AG012     Deposit  250000.0 2026-03-02           False   
315  TX00316    AG014     Deposit  250000.0 2026-04-01           False   
424  TX00425    AG005  Withdrawal  250000.0 2026-03-02           False   
463  TX00464    AG009  Withdrawal  250000.0 2026-04-11           False   
594  TX00595    AG016     Deposit  250000.0 2026-03-11           False   

     is_outlier   county   onboarded  
306        True   Kisumu  2024-12-01  
315        True  Nairobi  2025-02-01  
424        True  Mombasa  2024-05-01  
463        True   Nakuru  2024-09-01  
594        True   Kiambu  2025-04-01  


In [17]:
county_summary = clean_df.groupby(['county', 'txn_type']).agg(
    total_value=('amount', 'sum'),
    num_transactions=('amount', 'count'),
    average_value=('amount', 'mean')
).reset_index()

print("=== COUNTY LEVEL SUMMARY ===")
display(county_summary)

clean_df['week'] = clean_df['txn_date'].dt.to_period('W').astype(str)
weekly_pivot = clean_df.pivot_table(
    index='county',
    columns='week',
    values='amount',
    aggfunc='sum',
    fill_value=0
)

print("=== WEEKLY TRANSACTION VALUE BY COUNTY ===")
display(weekly_pivot)

=== COUNTY LEVEL SUMMARY ===


,county,txn_type,total_value,num_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77024.5,27,2852.759259
2,Kisumu,Deposit,436501.5,74,5898.668919
3,Kisumu,Withdrawal,76348.5,37,2063.472973
4,Mombasa,Deposit,149094.0,72,2070.750000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543698.0,118,4607.610169
7,Nairobi,Withdrawal,161932.0,62,2611.806452
8,Nakuru,Deposit,184944.0,60,3082.400000
9,Nakuru,Withdrawal,328568.5,39,8424.833333


=== WEEKLY TRANSACTION VALUE BY COUNTY ===


week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267745.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28126.0,27077.0,41077.0,15341.0,29470.0,37439.0,21308.0
Mombasa,6759.0,285696.5,30504.5,25171.5,17050.0,38344.0,47119.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52887.5,43134.5,41400.5,282931.0,56813.5,56406.5,65135.0,30908.5
Nakuru,2770.5,56574.0,28730.0,39795.0,18972.5,18145.0,276609.0,26179.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


In [18]:
import pandas as pd
import numpy as np

# Let's perform a comprehensive manual audit to document the data quality issues
# We will construct a DataFrame representing the specific quality audit requested.
audit_records = []

# Issue 1: Uniqueness (duplicates)
duplicate_rows = txns.duplicated(subset=['txn_id']).sum()
audit_records.append({
    'Issue': 'Duplicate transaction IDs found',
    'Dimension': 'Uniqueness',
    'Column': 'txn_id',
    'Rows Affected': duplicate_rows
})

# Issue 2: Consistency (lowercase / inconsistent names for deposit / withdrawal)
raw_types = txns['txn_type'].unique()
inconsistent_types_count = txns[~txns['txn_type'].isin(['Deposit', 'Withdrawal'])].shape[0]
audit_records.append({
    'Issue': 'Inconsistent transaction type formats (whitespace or mixed-case)',
    'Dimension': 'Consistency',
    'Column': 'txn_type',
    'Rows Affected': inconsistent_types_count
})

# Issue 3: Missing/Invalid amount values
missing_or_na_amounts = txns['amount'].isna().sum() + (txns['amount'] == 'N/A').sum()
audit_records.append({
    'Issue': 'Missing or "N/A" values in transaction amounts',
    'Dimension': 'Completeness',
    'Column': 'amount',
    'Rows Affected': missing_or_na_amounts
})

# Issue 4: Inconsistent Date formats
date_format_issues = txns['txn_date'].str.contains('/').sum()
audit_records.append({
    'Issue': 'Dates formatted with "/" instead of "-"',
    'Dimension': 'Validity',
    'Column': 'txn_date',
    'Rows Affected': date_format_issues
})

# Issue 5: Unknown agents referenced in transaction logs
unknown_agents_total = (~txns['agent_id'].isin(agents['agent_id'])).sum()
audit_records.append({
    'Issue': 'Transactions referencing agent IDs not in registration list',
    'Dimension': 'Validity',
    'Column': 'agent_id',
    'Rows Affected': unknown_agents_total
})

audit_df = pd.DataFrame(audit_records)
display(audit_df)


,Issue,Dimension,Column,Rows Affected
0,Duplicate transaction IDs found,Uniqueness,txn_id,12
1,Inconsistent transaction type formats (whitesp...,Consistency,txn_type,384
2,"Missing or ""N/A"" values in transaction amounts",Completeness,amount,22
3,"Dates formatted with ""/"" instead of ""-""",Validity,txn_date,5
4,Transactions referencing agent IDs not in regi...,Validity,agent_id,8


In [19]:
# Outliers analysis
outliers_only = clean_df[clean_df['is_outlier']]
print(f"Number of outliers flagged (IQR method): {len(outliers_only)}")
print("Investigation of KES 250,000 values:")
print(clean_df[clean_df['amount'] == 250000])


Number of outliers flagged (IQR method): 53
Investigation of KES 250,000 values:
      txn_id agent_id    txn_type    amount   txn_date  amount_imputed  \
306  TX00307    AG012     Deposit  250000.0 2026-03-02           False   
315  TX00316    AG014     Deposit  250000.0 2026-04-01           False   
424  TX00425    AG005  Withdrawal  250000.0 2026-03-02           False   
463  TX00464    AG009  Withdrawal  250000.0 2026-04-11           False   
594  TX00595    AG016     Deposit  250000.0 2026-03-11           False   

     is_outlier   county   onboarded                   week  
306        True   Kisumu  2024-12-01  2026-03-02/2026-03-08  
315        True  Nairobi  2025-02-01  2026-03-30/2026-04-05  
424        True  Mombasa  2024-05-01  2026-03-02/2026-03-08  
463        True   Nakuru  2024-09-01  2026-04-06/2026-04-12  
594        True   Kiambu  2025-04-01  2026-03-09/2026-03-15  


In [20]:
# Group and summarize county metrics
# (a) County-level summary
county_summary = clean_df.groupby(['county', 'txn_type']).agg(
    total_value=('amount', 'sum'),
    num_transactions=('amount', 'count'),
    average_value=('amount', 'mean')
).reset_index()

print("=== COUNTY LEVEL SUMMARY ===")
display(county_summary)

# (b) Weekly transaction pivot table by county
clean_df['week'] = clean_df['txn_date'].dt.to_period('W').astype(str)
weekly_pivot = clean_df.pivot_table(
    index='county',
    columns='week',
    values='amount',
    aggfunc='sum',
    fill_value=0
)

print("=== WEEKLY TRANSACTION VALUE BY COUNTY ===")
display(weekly_pivot)


=== COUNTY LEVEL SUMMARY ===


,county,txn_type,total_value,num_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77024.5,27,2852.759259
2,Kisumu,Deposit,436501.5,74,5898.668919
3,Kisumu,Withdrawal,76348.5,37,2063.472973
4,Mombasa,Deposit,149094.0,72,2070.750000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543698.0,118,4607.610169
7,Nairobi,Withdrawal,161932.0,62,2611.806452
8,Nakuru,Deposit,184944.0,60,3082.400000
9,Nakuru,Withdrawal,328568.5,39,8424.833333


=== WEEKLY TRANSACTION VALUE BY COUNTY ===


week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267745.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28126.0,27077.0,41077.0,15341.0,29470.0,37439.0,21308.0
Mombasa,6759.0,285696.5,30504.5,25171.5,17050.0,38344.0,47119.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52887.5,43134.5,41400.5,282931.0,56813.5,56406.5,65135.0,30908.5
Nakuru,2770.5,56574.0,28730.0,39795.0,18972.5,18145.0,276609.0,26179.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0
